# IFRS S1/S2 + Commercial Banks Requirement Extraction — Azure REST version

This notebook extracts structured disclosure requirements from the **three sources you need for the bank reporting project**:

1. **IFRS S1** — General Requirements for Disclosure of Sustainability-related Financial Information
2. **IFRS S2** — Climate-related Disclosures
3. **IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks**

This version uses the same Azure REST logic as your working governance notebook:
- direct REST calls with `urllib.request`
- full Azure chat-completions deployment URLs
- `.env` loading with `python-dotenv`
- JSON mode for extraction
- retry + token-field fallback between `max_completion_tokens` and `max_tokens`

It is designed to be run **cell by cell** so you can debug each step:
1. Install/import packages
2. Configure the 3 source documents
3. Load Azure REST configuration from `.env`
4. Parse PDFs
5. Create chunks
6. Preview chunks before sending them to the model
7. Test extraction on one chunk
8. Run full extraction with checkpointing
9. Deduplicate, validate, and export
10. Generate a reusable `requirements_kb.py`
11. Test retrieval by report section
12. Plug the requirements into a section-agent prompt

Important: the Commercial Banks file is **industry-based guidance that accompanies IFRS S2**. It suggests ways to apply IFRS S2 for commercial banks, but it does not create extra IFRS S2 requirements. The notebook therefore labels it separately as `S2_IBG_CB`.


## 0. Folder structure expected

Create this structure before running the full extraction:

```text
project/
│
├── data/
│   ├── standards/
│   │   ├── ifrs_s1.pdf
│   │   ├── ifrs_s2.pdf
│   │   └── ifrs_s2_ibg_volume_16_commercial_banks.pdf
│   │
│   └── requirements/
│
└── notebooks/
    └── 01_extract_ifrs_requirements_S1_S2_CommercialBanks_AZURE_REST.ipynb
```

You can rename the PDF files, but if you do, update the `INPUT_PDFS` list in the configuration cell.

Recommended naming:

```text
ifrs_s1.pdf
ifrs_s2.pdf
ifrs_s2_ibg_volume_16_commercial_banks.pdf
```


In [ ]:
# 1. Install packages
# Run this once in your notebook environment.

%pip install -q --upgrade pydantic pypdf tenacity pandas python-dotenv


In [ ]:
# 2. Imports

import os
import re
import json
import time
import random
import urllib.request
import urllib.error
from pathlib import Path
from collections import defaultdict, Counter
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError
from pypdf import PdfReader
from tenacity import retry, stop_after_attempt, wait_exponential
from dotenv import load_dotenv, find_dotenv

print("Imports loaded successfully.")


In [ ]:
# 3. Configuration

# This notebook uses Azure REST chat-completions URLs, so there is no client-side model name.
# The deployment/model is already inside your full Azure URL.
#
# Recommended .env variables:
#   AZURE_OPENAI_API_KEY=<shared Azure resource key>
#   AZURE_OPENAI_EXTRACTOR_URL=<full GPT-5.2 chat-completions deployment URL>
#
# Fallback supported:
#   If AZURE_OPENAI_EXTRACTOR_URL is missing, the notebook uses AZURE_OPENAI_JUDGE_URL.
#   If AZURE_OPENAI_EXTRACTOR_API_KEY is missing, the notebook uses AZURE_OPENAI_API_KEY or AZURE_OPENAI_JUDGE_API_KEY.

MODEL_LABEL = "Azure GPT-5.2 extraction deployment"

# Use smaller chunks while debugging. Increase later if needed.
MAX_CHARS_PER_CHUNK = 12_000

# Set this to a small number while testing. Set to None for all chunks.
MAX_CHUNKS_PER_DOCUMENT = None

# Maximum output tokens for one extraction call.
# If you get truncated JSON, increase this.
EXTRACTION_MAX_OUTPUT_TOKENS = 5000

# Output folders
BASE_DIR = Path(".")
STANDARDS_DIR = BASE_DIR / "data" / "standards"
OUTPUT_DIR = BASE_DIR / "data" / "requirements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = OUTPUT_DIR / "ifrs_bank_requirements.json"
OUTPUT_RAW_JSONL = OUTPUT_DIR / "ifrs_bank_requirements_raw_checkpoint.jsonl"
OUTPUT_KB = OUTPUT_DIR / "requirements_kb.py"

# Helper for file names.
# It lets the notebook accept either the clean recommended name or the original uploaded/downloaded name.
def first_existing_path(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    # Return the preferred first path so the missing-file message is clear.
    return candidates[0]


# Your 3 source PDFs only:
# 1) IFRS S1
# 2) IFRS S2
# 3) IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks
INPUT_PDFS = [
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s1.pdf",
            STANDARDS_DIR / "ifrs-s1-general-requirements.pdf",
        ),
        "standard": "S1",
        "source_doc": "IFRS S1 General Requirements",
        "source_authority": "core_standard",
    },
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s2.pdf",
            STANDARDS_DIR / "ifrs-s2-climate-related-disclosures.pdf",
        ),
        "standard": "S2",
        "source_doc": "IFRS S2 Climate-related Disclosures",
        "source_authority": "core_standard",
    },
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s2_ibg_volume_16_commercial_banks.pdf",
            STANDARDS_DIR / "ifrs-s2-ibg-volume-16-commercial-banks-part-b.pdf",
            STANDARDS_DIR / "ifrs-s2-ibg-volume-16-commercial-banks-part-b (1).pdf",
        ),
        "standard": "S2_IBG_CB",
        "source_doc": "IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks",
        "source_authority": "industry_guidance",
    },
]

print("Model/deployment label:", MODEL_LABEL)
print("Standards folder:", STANDARDS_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Chunk size:", MAX_CHARS_PER_CHUNK)

print("\nConfigured source documents:")
for doc in INPUT_PDFS:
    print(f"- {doc['standard']}: {doc['source_doc']} -> {doc['path']}")


In [ ]:
# 4. Check that your PDFs exist

missing = []

for item in INPUT_PDFS:
    path = item["path"]
    if path.exists():
        print(f"FOUND: {path}")
    else:
        print(f"MISSING: {path}")
        missing.append(path)

if missing:
    print("\nSome PDFs are missing. Add them to data/standards/ or update INPUT_PDFS.")
else:
    print("\nAll PDFs found.")


In [ ]:
# 5. Azure OpenAI REST setup
# This cell matches the logic style of your working governance notebook.

# Safety defaults in case this cell is run before the configuration cell.
# The config cell values still take priority when already defined.
EXTRACTION_MAX_OUTPUT_TOKENS = globals().get("EXTRACTION_MAX_OUTPUT_TOKENS", 5000)
MAX_CHARS_PER_CHUNK = globals().get("MAX_CHARS_PER_CHUNK", 12000)
MAX_CHUNKS_PER_DOCUMENT = globals().get("MAX_CHUNKS_PER_DOCUMENT", None)

print("Runtime defaults checked:")
print("EXTRACTION_MAX_OUTPUT_TOKENS =", EXTRACTION_MAX_OUTPUT_TOKENS)
print("MAX_CHARS_PER_CHUNK =", MAX_CHARS_PER_CHUNK)

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


# Shared-key fallback.
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

# For this notebook, extraction should usually use GPT-5.2.
# You can either define AZURE_OPENAI_EXTRACTOR_URL directly,
# or reuse your existing AZURE_OPENAI_JUDGE_URL from the governance notebook.
AZURE_OPENAI_EXTRACTOR_API_KEY = (
    os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY")
    or _shared_key_fallback
)

AZURE_OPENAI_EXTRACTOR_URL = _clean_url(
    os.getenv("AZURE_OPENAI_EXTRACTOR_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)


def validate_extractor_config() -> None:
    missing = []

    if not AZURE_OPENAI_EXTRACTOR_API_KEY:
        missing.append("AZURE_OPENAI_EXTRACTOR_API_KEY or AZURE_OPENAI_API_KEY")

    if not AZURE_OPENAI_EXTRACTOR_URL:
        missing.append("AZURE_OPENAI_EXTRACTOR_URL or AZURE_OPENAI_JUDGE_URL")

    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "extractor_key_loaded": bool(AZURE_OPENAI_EXTRACTOR_API_KEY),
            "judge_key_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_API_KEY")),
            "extractor_url_loaded": bool(os.getenv("AZURE_OPENAI_EXTRACTOR_URL")),
            "judge_url_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_URL")),
            "chat_url_loaded": bool(os.getenv("AZURE_OPENAI_CHAT_URL")),
        }

        raise ValueError(
            "Missing Azure extractor configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags, keys are never printed:\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nExpected .env example:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_EXTRACTOR_URL=<full GPT-5.2 chat-completions deployment URL>\n\n"
              "Alternative using your existing governance setup:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 chat-completions deployment URL>"
        )

    if not AZURE_OPENAI_EXTRACTOR_URL.startswith("https://"):
        raise ValueError(
            "AZURE_OPENAI_EXTRACTOR_URL must be a full HTTPS Azure deployment URL. "
            f"Current value: {AZURE_OPENAI_EXTRACTOR_URL!r}"
        )

    if "/chat/completions" not in AZURE_OPENAI_EXTRACTOR_URL:
        print("WARNING: The extractor URL does not contain '/chat/completions'.")
        print("Expected full URL format:")
        print("https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=<version>")


validate_extractor_config()

print("Azure OpenAI extractor configuration loaded.")
print("Extractor endpoint:", AZURE_OPENAI_EXTRACTOR_URL[:110] + "...")
print("API key loaded:", bool(AZURE_OPENAI_EXTRACTOR_API_KEY))


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = True,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 4,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway rejects it, retries using `max_tokens`, because some
      enterprise proxies do not yet forward `max_completion_tokens` correctly.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )

    token_fields = [preferred_field]

    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_extractor_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 extractor with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Requires a JSON object with a top-level `requirements` array.

    On invalid/truncated JSON:
    - Sends the returned content back to the same Azure endpoint for JSON repair.
    """

    data = _azure_chat_completion(
        url=AZURE_OPENAI_EXTRACTOR_URL,
        api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=EXTRACTION_MAX_OUTPUT_TOKENS,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=True,
        request_label="GPT-5.2 IFRS extractor",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("Extractor returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "The object must contain a top-level key named requirements. "
            "The value must be an array. "
            "Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated extractor output into one complete valid JSON object.

Required schema:
{{
  "requirements": [
    {{
      "standard": "S1 or S2 or S2_IBG_CB",
      "source_doc": "string",
      "source_authority": "core_standard or industry_guidance",
      "paragraph": "string",
      "section": "general_requirements or governance or strategy or risk_management or metrics_targets or industry_metrics or other",
      "obligation_type": "shall or should or may",
      "requirement_text": "string",
      "applies_to_banks": true,
      "related_paragraphs": [],
      "metric_type": null,
      "page_start": 1,
      "page_end": 1
    }}
  ]
}}

Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_EXTRACTOR_URL,
            api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=EXTRACTION_MAX_OUTPUT_TOKENS,
            use_max_completion_tokens=True,
            temperature=None,
            json_mode=True,
            request_label="GPT-5.2 IFRS extractor JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "Extractor failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


print("Azure REST helper functions ready.")


In [ ]:
# 5B. Optional smoke test: verify the Azure extractor endpoint responds

# This cell calls Azure once with a tiny JSON-mode request.
# Run this before the real extraction if you want to confirm the endpoint works.

smoke_result = call_extractor_llm_json(
    system_prompt="You return valid JSON only.",
    user_prompt='Return exactly this JSON object: {"requirements": []}'
)

print(smoke_result)


In [ ]:
# 6. Structured output schema

class IFRSRequirement(BaseModel):
    model_config = ConfigDict(extra="forbid")

    # S2_IBG_CB = IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks
    standard: Literal["S1", "S2", "S2_IBG_CB"]
    source_doc: str
    source_authority: Literal["core_standard", "industry_guidance"]

    paragraph: str
    section: Literal[
        "general_requirements",
        "governance",
        "strategy",
        "risk_management",
        "metrics_targets",
        "industry_metrics",
        "other",
    ]

    obligation_type: Literal["shall", "should", "may"]

    requirement_text: str
    applies_to_banks: bool
    related_paragraphs: list[str]
    metric_type: str | None

    page_start: int
    page_end: int


class RequirementExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    requirements: list[IFRSRequirement]


print("Pydantic schemas ready.")


In [ ]:
# 7. PDF parsing helpers

def clean_text(text: str) -> str:
    """Basic PDF text cleanup."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """Extract text page by page from a PDF."""
    reader = PdfReader(str(pdf_path))
    pages = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = clean_text(text)

        if text:
            pages.append({
                "page": i,
                "text": text,
            })

    return pages


def make_chunks(pages: list[dict], max_chars: int = MAX_CHARS_PER_CHUNK) -> list[dict]:
    """Group PDF pages into chunks while preserving page range metadata."""
    chunks = []
    current_text = []
    current_pages = []
    current_len = 0

    for page in pages:
        page_text = f"\n\n[PAGE {page['page']}]\n{page['text']}"

        if current_text and current_len + len(page_text) > max_chars:
            chunks.append({
                "text": "\n".join(current_text),
                "page_start": min(current_pages),
                "page_end": max(current_pages),
                "char_count": current_len,
            })

            current_text = []
            current_pages = []
            current_len = 0

        current_text.append(page_text)
        current_pages.append(page["page"])
        current_len += len(page_text)

    if current_text:
        chunks.append({
            "text": "\n".join(current_text),
            "page_start": min(current_pages),
            "page_end": max(current_pages),
            "char_count": current_len,
        })

    return chunks


print("PDF parsing helpers ready.")


In [ ]:
# 8. Test PDF parsing on the first available PDF

available_pdfs = [x for x in INPUT_PDFS if x["path"].exists()]

if not available_pdfs:
    raise FileNotFoundError("No PDFs found. Add PDFs to data/standards/ first.")

test_pdf = available_pdfs[0]
pages = extract_pdf_pages(test_pdf["path"])
chunks = make_chunks(pages)

print("Test document:", test_pdf["source_doc"])
print("Pages extracted:", len(pages))
print("Chunks created:", len(chunks))

if pages:
    print("\nFirst page preview:")
    print(pages[0]["text"][:1500])


In [ ]:
# 9. Preview chunks before sending to GPT

chunk_preview_rows = []

for pdf_info in available_pdfs:
    pages = extract_pdf_pages(pdf_info["path"])
    chunks = make_chunks(pages)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_preview_rows.append({
            "source_doc": pdf_info["source_doc"],
            "standard": pdf_info["standard"],
            "source_authority": pdf_info["source_authority"],
            "chunk_index": idx,
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "char_count": chunk["char_count"],
            "text_preview": chunk["text"][:250].replace("\n", " "),
        })

chunk_df = pd.DataFrame(chunk_preview_rows)
chunk_df.head(10)


In [ ]:
# 10. Prompts

SYSTEM_PROMPT = """
You are an IFRS S1/S2 and banking disclosure requirement extraction specialist.

Your job is to extract structured disclosure requirements from three source types:

1. IFRS S1 core standard
2. IFRS S2 core standard
3. IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks

Important source authority distinction:
- For S1 and S2, extract mandatory disclosure requirements as core standard requirements.
- For S2_IBG_CB, extract the Commercial Banks industry-based guidance as industry guidance.
  This guidance accompanies IFRS S2 and suggests ways to apply IFRS S2 for commercial banks.
  It does not create additional IFRS S2 requirements, but it is still important because IFRS S2 requires entities to refer to and consider applicable industry-based guidance.

Extract only actual disclosure obligations, application guidance, technical protocols, metrics, or activity metrics that affect disclosure.

A requirement or guidance item is usually indicated by wording such as:
- shall disclose
- shall include
- shall describe
- shall provide
- shall explain
- is required to disclose
- should disclose
- may disclose
- metric
- activity metric
- technical protocol
- scope of disclosure

Do not invent requirements.
Do not paraphrase the paragraph text.
The requirement_text must be copied from the provided text as closely as possible.

Classify each item into one section:
- general_requirements
- governance
- strategy
- risk_management
- metrics_targets
- industry_metrics
- other

Classification guidance:
- Use industry_metrics for Commercial Banks FN-CB metrics and activity metrics.
- Use risk_management for credit analysis, ESG integration in lending, portfolio risk, scenario analysis, and credit exposure concentration.
- Use metrics_targets for cross-industry climate metrics such as GHG emissions, Scope 1/2/3, targets, capital deployment, physical risk, transition risk, and climate opportunities.
- Use governance for board oversight, management roles, controls, oversight and accountability.
- Use strategy for business model, value chain, financial effects, resilience, transition plans, and climate-related opportunities.
- Use general_requirements for materiality, reporting entity, timing, location, comparative information, judgement, estimation uncertainty and general presentation requirements.

Set applies_to_banks to true when:
- the standard is S2_IBG_CB; or
- the text refers to banks, commercial banks, financial institutions, lending, loans, project finance, credit analysis, credit exposure, financed emissions, borrowers, collateral, asset class, portfolios, or similar banking concepts.

You must return a valid JSON object only.
The JSON object must have exactly this top-level structure:
{
  "requirements": [...]
}

Do not return a bare array.
Do not include markdown fences.
Do not include commentary.
""".strip()


def build_user_prompt(
    text_chunk: str,
    standard: str,
    source_doc: str,
    source_authority: str,
    page_start: int,
    page_end: int,
) -> str:
    return f"""
STANDARD: {standard}
SOURCE DOCUMENT: {source_doc}
SOURCE AUTHORITY: {source_authority}
PAGE RANGE: {page_start}-{page_end}

Extract every relevant disclosure requirement, guidance item, metric, activity metric, or technical protocol from the text below.

For every extracted item:
- standard must be "{standard}"
- source_doc must be "{source_doc}"
- source_authority must be "{source_authority}"
- page_start must be {page_start}
- page_end must be {page_end}
- paragraph must be the paragraph number, metric code, or guidance code if visible, for example:
  - "29"
  - "B63"
  - "FN-CB-410a.2"
  - "FN-CB-000.B"
  - "unknown"
- obligation_type must be "shall", "should", or "may"
  - Use "shall" for mandatory wording or metric protocols that state "the entity shall..."
  - Use "should" for recommendations or "should" wording
  - Use "may" for permitted optional disclosures or "may" wording
- related_paragraphs must be an empty list if no cross-reference is present
- metric_type must be null if no specific metric is required
- For FN-CB metrics, metric_type should contain the metric code and short metric name where possible
- applies_to_banks must be true for all S2_IBG_CB items

Return one JSON object only, with this shape:
{{
  "requirements": [
    {{
      "standard": "{standard}",
      "source_doc": "{source_doc}",
      "source_authority": "{source_authority}",
      "paragraph": "string",
      "section": "general_requirements|governance|strategy|risk_management|metrics_targets|industry_metrics|other",
      "obligation_type": "shall|should|may",
      "requirement_text": "string",
      "applies_to_banks": true,
      "related_paragraphs": [],
      "metric_type": null,
      "page_start": {page_start},
      "page_end": {page_end}
    }}
  ]
}}

If the text contains no relevant disclosure requirements or guidance items, return:
{{"requirements": []}}

TEXT:
{text_chunk}
""".strip()


print("Prompts ready.")


In [ ]:
# 11. GPT extraction function for one chunk
# This uses the Azure REST helper from Cell 5 instead of OpenAI responses.parse.

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=20))
def extract_requirements_from_chunk(
    text_chunk: str,
    standard: str,
    source_doc: str,
    source_authority: str,
    page_start: int,
    page_end: int,
) -> list[dict]:
    """Extract requirements from one text chunk using Azure REST chat completions."""

    result = call_extractor_llm_json(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=build_user_prompt(
            text_chunk=text_chunk,
            standard=standard,
            source_doc=source_doc,
            source_authority=source_authority,
            page_start=page_start,
            page_end=page_end,
        ),
    )

    if not isinstance(result, dict):
        raise ValueError(
            "Extractor output must be a JSON object, but got: "
            + type(result).__name__
        )

    reqs = result.get("requirements", [])

    if reqs is None:
        reqs = []

    if not isinstance(reqs, list):
        raise ValueError(
            "Extractor output field `requirements` must be a list. Output preview:\n"
            + json.dumps(result, indent=2, ensure_ascii=False)[:3000]
        )

    cleaned = []

    for raw_req in reqs:
        # Force metadata to the chunk values. This prevents small model mistakes.
        raw_req["standard"] = standard
        raw_req["source_doc"] = source_doc
        raw_req["source_authority"] = source_authority
        raw_req["page_start"] = page_start
        raw_req["page_end"] = page_end

        # Commercial Banks industry guidance is bank-specific by definition.
        if standard == "S2_IBG_CB":
            raw_req["applies_to_banks"] = True

        # Normalize empty/invalid fields before Pydantic validation.
        if not raw_req.get("paragraph"):
            raw_req["paragraph"] = "unknown"

        if raw_req.get("metric_type") in ["", "none", "None", "null", "NULL"]:
            raw_req["metric_type"] = None

        if not isinstance(raw_req.get("related_paragraphs"), list):
            raw_req["related_paragraphs"] = []

        validated = IFRSRequirement.model_validate(raw_req)
        cleaned.append(validated.model_dump())

    return cleaned


print("Azure REST extraction function ready.")


In [ ]:
# 12. DEBUG: run extraction on only one chunk first

# This cell calls the Azure OpenAI API.
# It is the best cell to debug model access, schema problems, and output quality.

pdf_info = available_pdfs[0]
pages = extract_pdf_pages(pdf_info["path"])
chunks = make_chunks(pages)

test_chunk_index = 0
test_chunk = chunks[test_chunk_index]

print("Testing on:")
print("Document:", pdf_info["source_doc"])
print("Standard:", pdf_info["standard"])
print("Authority:", pdf_info["source_authority"])
print("Chunk:", test_chunk_index + 1)
print("Pages:", test_chunk["page_start"], "-", test_chunk["page_end"])
print("Characters:", test_chunk["char_count"])

test_requirements = extract_requirements_from_chunk(
    text_chunk=test_chunk["text"],
    standard=pdf_info["standard"],
    source_doc=pdf_info["source_doc"],
    source_authority=pdf_info["source_authority"],
    page_start=test_chunk["page_start"],
    page_end=test_chunk["page_end"],
)

print(f"Requirements found: {len(test_requirements)}")

pd.DataFrame(test_requirements).head(20)


In [ ]:
# 13. Inspect one extracted requirement in full

if not test_requirements:
    print("No requirements found in the test chunk.")
else:
    i = 0
    print(json.dumps(test_requirements[i], indent=2, ensure_ascii=False))


In [ ]:
# 14. Checkpoint helpers

def chunk_key(source_doc: str, chunk_index: int, page_start: int, page_end: int) -> str:
    return f"{source_doc}::chunk={chunk_index}::pages={page_start}-{page_end}"


def load_processed_chunk_keys(checkpoint_path: Path) -> set[str]:
    """Load chunk keys that were already processed."""
    if not checkpoint_path.exists():
        return set()

    keys = set()
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            keys.add(record["chunk_key"])

    return keys


def append_checkpoint_records(checkpoint_path: Path, key: str, requirements: list[dict]) -> None:
    """Append extracted requirements for one chunk to a JSONL checkpoint file."""
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for req in requirements:
            f.write(json.dumps({
                "chunk_key": key,
                "requirement": req,
            }, ensure_ascii=False) + "\n")


def load_checkpoint_requirements(checkpoint_path: Path) -> list[dict]:
    """Load all requirements from the JSONL checkpoint file."""
    if not checkpoint_path.exists():
        return []

    requirements = []
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            requirements.append(record["requirement"])

    return requirements


print("Checkpoint helpers ready.")


In [ ]:
# 15. FULL RUN: extract all requirements with checkpointing

# This cell calls the Azure OpenAI API many times.
# While debugging, set MAX_CHUNKS_PER_DOCUMENT = 1 or 2 in the config cell.
# When you are confident, set MAX_CHUNKS_PER_DOCUMENT = None and rerun.

processed_keys = load_processed_chunk_keys(OUTPUT_RAW_JSONL)
print(f"Already processed chunks in checkpoint: {len(processed_keys)}")

total_new_chunks = 0

for pdf_info in available_pdfs:
    pdf_path = pdf_info["path"]
    standard = pdf_info["standard"]
    source_doc = pdf_info["source_doc"]
    source_authority = pdf_info["source_authority"]

    print("\n" + "=" * 80)
    print(f"Processing: {source_doc}")
    print(f"Standard: {standard}")
    print(f"Authority: {source_authority}")
    print(f"File: {pdf_path}")

    pages = extract_pdf_pages(pdf_path)
    chunks = make_chunks(pages)

    if MAX_CHUNKS_PER_DOCUMENT is not None:
        chunks = chunks[:MAX_CHUNKS_PER_DOCUMENT]

    print(f"Pages extracted: {len(pages)}")
    print(f"Chunks to process: {len(chunks)}")

    for idx, chunk in enumerate(chunks, start=1):
        key = chunk_key(source_doc, idx, chunk["page_start"], chunk["page_end"])

        if key in processed_keys:
            print(f"Skipping already processed chunk {idx}/{len(chunks)} pages {chunk['page_start']}-{chunk['page_end']}")
            continue

        print(f"Extracting chunk {idx}/{len(chunks)} pages {chunk['page_start']}-{chunk['page_end']}")

        try:
            reqs = extract_requirements_from_chunk(
                text_chunk=chunk["text"],
                standard=standard,
                source_doc=source_doc,
                source_authority=source_authority,
                page_start=chunk["page_start"],
                page_end=chunk["page_end"],
            )

            append_checkpoint_records(OUTPUT_RAW_JSONL, key, reqs)
            processed_keys.add(key)
            total_new_chunks += 1

            print(f"  Found {len(reqs)} requirements")

            # Gentle pause to reduce rate-limit pressure.
            time.sleep(0.5)

        except Exception as e:
            print(f"  ERROR on chunk {idx}: {type(e).__name__}: {e}")
            print("  You can fix the issue and rerun this cell; checkpointed chunks will be skipped.")
            raise

print("\nFull run complete.")
print("New chunks processed:", total_new_chunks)
print("Checkpoint file:", OUTPUT_RAW_JSONL)


In [ ]:
# 16. Load raw checkpoint results

raw_requirements = load_checkpoint_requirements(OUTPUT_RAW_JSONL)

print("Raw extracted requirements:", len(raw_requirements))

if raw_requirements:
    raw_df = pd.DataFrame(raw_requirements)
    display(raw_df.head(20))
else:
    print("No checkpoint requirements found yet.")


In [ ]:
# 17. Deduplication and sorting

def normalize_text_for_dedupe(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def deduplicate_requirements(requirements: list[dict]) -> list[dict]:
    seen = set()
    deduped = []

    for req in requirements:
        key = (
            req.get("standard"),
            req.get("paragraph"),
            normalize_text_for_dedupe(req.get("requirement_text", ""))[:350],
        )

        if key not in seen:
            seen.add(key)
            deduped.append(req)

    return deduped


def sort_requirements(requirements: list[dict]) -> list[dict]:
    section_order = {
        "general_requirements": 0,
        "governance": 1,
        "strategy": 2,
        "risk_management": 3,
        "metrics_targets": 4,
        "industry_metrics": 5,
        "other": 6,
    }

    obligation_order = {
        "shall": 0,
        "should": 1,
        "may": 2,
    }

    return sorted(
        requirements,
        key=lambda r: (
            r.get("standard", ""),
            section_order.get(r.get("section", "other"), 99),
            obligation_order.get(r.get("obligation_type", "may"), 99),
            r.get("page_start", 999999),
            str(r.get("paragraph", "")),
        ),
    )


deduped_requirements = deduplicate_requirements(raw_requirements)
final_requirements = sort_requirements(deduped_requirements)

print("Raw requirements:", len(raw_requirements))
print("After deduplication:", len(final_requirements))


In [ ]:
# 18. Validate final requirements against the schema

valid_requirements = []
invalid_requirements = []

for req in final_requirements:
    try:
        valid_req = IFRSRequirement.model_validate(req)
        valid_requirements.append(valid_req.model_dump())
    except ValidationError as e:
        invalid_requirements.append({
            "requirement": req,
            "errors": e.errors(),
        })

print("Valid requirements:", len(valid_requirements))
print("Invalid requirements:", len(invalid_requirements))

if invalid_requirements:
    print("\nExample invalid requirement:")
    print(json.dumps(invalid_requirements[0], indent=2, ensure_ascii=False))


In [ ]:
# 19. Save final JSON

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(valid_requirements, f, ensure_ascii=False, indent=2)

print(f"Saved final requirements JSON to: {OUTPUT_JSON.resolve()}")
print(f"Total saved requirements: {len(valid_requirements)}")


In [ ]:
# 20. Quality-control summaries

if not valid_requirements:
    print("No valid requirements to summarize yet.")
else:
    df = pd.DataFrame(valid_requirements)

    print("By section:")
    display(df["section"].value_counts().rename_axis("section").reset_index(name="count"))

    print("By standard:")
    display(df["standard"].value_counts().rename_axis("standard").reset_index(name="count"))

    print("By source authority:")
    display(df["source_authority"].value_counts().rename_axis("source_authority").reset_index(name="count"))

    print("By obligation type:")
    display(df["obligation_type"].value_counts().rename_axis("obligation_type").reset_index(name="count"))

    print("Bank-specific requirements:")
    display(df["applies_to_banks"].value_counts().rename_axis("applies_to_banks").reset_index(name="count"))

    display(df.head(20))


In [ ]:
# 21. Inspect requirements by section

SECTION_TO_INSPECT = "governance"

section_reqs = [
    r for r in valid_requirements
    if r["section"] == SECTION_TO_INSPECT
]

print(f"Section: {SECTION_TO_INSPECT}")
print(f"Requirements: {len(section_reqs)}")

for r in section_reqs[:10]:
    print("\n" + "-" * 80)
    print(f"{r['standard']} ¶{r['paragraph']} | {r['obligation_type']} | bank={r['applies_to_banks']}")
    print(r["requirement_text"][:1000])


In [ ]:
# 22. Generate requirements_kb.py

def write_requirements_kb(requirements: list[dict], output_path: Path) -> None:
    grouped = defaultdict(list)

    for req in requirements:
        grouped[req["section"]].append(req)

    grouped = dict(grouped)

    kb_code = f'''"""
Auto-generated IFRS S1/S2 + Commercial Banks requirements knowledge base.
Do not edit manually. Regenerate from the extraction notebook.

Sources:
- IFRS S1 General Requirements
- IFRS S2 Climate-related Disclosures
- IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks
"""

REQUIREMENTS = {repr(grouped)}


def get_requirements(
    section: str,
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
) -> list[dict]:
    """Return requirements by section, optionally filtered by standard/authority/bank relevance."""

    reqs = REQUIREMENTS.get(section, [])

    if standard:
        reqs = [r for r in reqs if r["standard"] == standard]

    if source_authority:
        reqs = [r for r in reqs if r["source_authority"] == source_authority]

    if mandatory_only:
        reqs = [r for r in reqs if r["obligation_type"] == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r["applies_to_banks"]]

    return sorted(
        reqs,
        key=lambda r: (
            r["source_authority"] != "core_standard",
            r["obligation_type"] != "shall",
            r["standard"],
            r["page_start"],
            r["paragraph"],
        )
    )


def get_core_requirements(section: str, mandatory_only: bool = True) -> list[dict]:
    """Return only IFRS S1/S2 core standard requirements for a section."""
    return get_requirements(
        section=section,
        source_authority="core_standard",
        mandatory_only=mandatory_only,
    )


def get_bank_guidance(section: str | None = None) -> list[dict]:
    """Return Commercial Banks industry guidance. If section is None, return all bank guidance."""
    if section is None:
        all_reqs = []
        for reqs in REQUIREMENTS.values():
            all_reqs.extend(reqs)
        return sorted(
            [r for r in all_reqs if r["standard"] == "S2_IBG_CB"],
            key=lambda r: (r["page_start"], r["paragraph"])
        )

    return get_requirements(
        section=section,
        standard="S2_IBG_CB",
        banks_only=True,
    )


def list_sections() -> list[str]:
    return sorted(REQUIREMENTS.keys())


def count_requirements() -> int:
    return sum(len(v) for v in REQUIREMENTS.values())
'''

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(kb_code)


write_requirements_kb(valid_requirements, OUTPUT_KB)

print(f"Saved Python KB to: {OUTPUT_KB.resolve()}")


In [ ]:
# 23. Test the generated KB without importing it

# This simulates how your report agents will query the KB.

requirements_by_section = defaultdict(list)

for req in valid_requirements:
    requirements_by_section[req["section"]].append(req)


def get_requirements_from_memory(
    section: str,
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
) -> list[dict]:
    reqs = list(requirements_by_section.get(section, []))

    if standard:
        reqs = [r for r in reqs if r["standard"] == standard]

    if source_authority:
        reqs = [r for r in reqs if r["source_authority"] == source_authority]

    if mandatory_only:
        reqs = [r for r in reqs if r["obligation_type"] == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r["applies_to_banks"]]

    return sorted(
        reqs,
        key=lambda r: (
            r["source_authority"] != "core_standard",
            r["obligation_type"] != "shall",
            r["standard"],
            r["page_start"],
            r["paragraph"],
        )
    )


test_governance_reqs = get_requirements_from_memory(
    section="governance",
    source_authority="core_standard",
    mandatory_only=True,
)

test_bank_guidance = get_requirements_from_memory(
    section="risk_management",
    standard="S2_IBG_CB",
    banks_only=True,
)

print("Mandatory core governance requirements:", len(test_governance_reqs))
print("Commercial Banks risk-management guidance items:", len(test_bank_guidance))

print("\nCore governance examples:")
for r in test_governance_reqs[:5]:
    print(f"- [{r['standard']} ¶{r['paragraph']}] {r['requirement_text'][:250]}...")

print("\nCommercial Banks guidance examples:")
for r in test_bank_guidance[:5]:
    print(f"- [{r['standard']} ¶{r['paragraph']}] {r['requirement_text'][:250]}...")


In [ ]:
# 24. Build a section-agent prompt using the extracted requirements

def build_section_prompt(section_name: str, payload: dict, requirements: list[dict]) -> str:
    core_reqs = [
        r for r in requirements
        if (
            r["section"] == section_name
            and r["source_authority"] == "core_standard"
            and r["obligation_type"] == "shall"
        )
    ]

    bank_guidance = [
        r for r in requirements
        if (
            r["section"] == section_name
            and r["standard"] == "S2_IBG_CB"
        )
    ]

    core_req_text = "\n".join(
        f"[{r['standard']} ¶{r['paragraph']}] {r['requirement_text']}"
        for r in core_reqs
    ) or "No core IFRS S1/S2 mandatory requirements were extracted for this section."

    bank_guidance_text = "\n".join(
        f"[{r['standard']} ¶{r['paragraph']}] {r['requirement_text']}"
        for r in bank_guidance
    ) or "No Commercial Banks industry guidance was extracted for this section."

    return f"""
You are writing the {section_name.replace("_", " ").title()} section of an IFRS S1/S2 sustainability report for a commercial bank.

CORE IFRS S1/S2 MANDATORY DISCLOSURE REQUIREMENTS:
You must address every applicable core requirement below.

{core_req_text}

COMMERCIAL BANKS INDUSTRY-BASED GUIDANCE:
Use this as banking-specific guidance to make the section relevant to commercial banking.
This guidance accompanies IFRS S2 and supports application for banks, but do not present it as creating separate IFRS requirements.

{bank_guidance_text}

BANK DATA:
{json.dumps(payload, indent=2, ensure_ascii=False)}

Instructions:
1. Write a professional sustainability disclosure section.
2. Use only the bank data provided.
3. Do not invent metrics.
4. If data is insufficient for a requirement or guidance item, explicitly state the missing data.
5. Reference IFRS paragraph numbers and FN-CB metric codes where relevant.
6. Keep core IFRS S1/S2 requirements separate from banking-specific guidance in your reasoning.
7. Keep the tone close to a real annual sustainability report.
""".strip()


# Example fake payload just to test prompt construction.
sample_governance_payload = {
    "bank_name": "Example Bank",
    "reporting_year": 2024,
    "board_oversight": {
        "board_climate_meetings": 6,
        "climate_reports_frequency": "semi-annual",
        "responsible_committee": "Risk Committee",
    },
    "management_role": {
        "executive_owner": "Chief Risk Officer",
        "climate_risk_team": "Enterprise Risk Management",
    },
}

governance_prompt = build_section_prompt(
    section_name="governance",
    payload=sample_governance_payload,
    requirements=valid_requirements,
)

print(governance_prompt[:4000])


In [ ]:
# 25. Optional: generate one report section with GPT
# This cell calls the same Azure extractor endpoint.
# Run it only after you are satisfied with the prompt preview above.
#
# Note: for final report writing, you may prefer your existing writer endpoint
# from the governance notebook. This cell is only a simple test.

def generate_section_draft(section_name: str, payload: dict, requirements: list[dict]) -> str:
    prompt = build_section_prompt(
        section_name=section_name,
        payload=payload,
        requirements=requirements,
    )

    data = _azure_chat_completion(
        url=AZURE_OPENAI_EXTRACTOR_URL,
        api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
        messages=[
            {
                "role": "system",
                "content": "You are an IFRS S1/S2 sustainability reporting specialist for banking.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_output_tokens=3000,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=False,
        request_label="GPT-5.2 section draft test",
    )

    return _extract_message_content(data)


# Uncomment to test:
# governance_draft = generate_section_draft(
#     section_name="governance",
#     payload=sample_governance_payload,
#     requirements=valid_requirements,
# )
#
# print(governance_draft)


In [ ]:
# 26. Build a simple coverage checklist for QA

def build_coverage_checklist(section_name: str, requirements: list[dict], include_guidance: bool = True) -> pd.DataFrame:
    section_reqs = []

    for r in requirements:
        if r["section"] != section_name:
            continue

        # Always include mandatory core IFRS requirements.
        if r["source_authority"] == "core_standard" and r["obligation_type"] == "shall":
            section_reqs.append(r)

        # Optionally include Commercial Banks industry guidance.
        elif include_guidance and r["standard"] == "S2_IBG_CB":
            section_reqs.append(r)

    rows = []

    for r in section_reqs:
        rows.append({
            "standard": r["standard"],
            "source_authority": r["source_authority"],
            "paragraph": r["paragraph"],
            "section": r["section"],
            "obligation_type": r["obligation_type"],
            "applies_to_banks": r["applies_to_banks"],
            "metric_type": r["metric_type"],
            "requirement_text": r["requirement_text"],
            "covered_in_draft": None,
            "evidence_sentence": None,
            "missing_data": None,
        })

    return pd.DataFrame(rows)


governance_checklist = build_coverage_checklist("governance", valid_requirements)
governance_checklist.head(20)


In [ ]:
# 27. Save checklist template

CHECKLIST_PATH = OUTPUT_DIR / "governance_coverage_checklist_template.csv"

if len(governance_checklist) > 0:
    governance_checklist.to_csv(CHECKLIST_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved checklist template to: {CHECKLIST_PATH.resolve()}")
else:
    print("No governance checklist rows to save yet.")


## How to use this notebook in your pipeline

After you run the full extraction, use these two files in your project:

```text
data/requirements/ifrs_bank_requirements.json
data/requirements/requirements_kb.py
```

Your section agents should receive:
1. The bank payload for the section
2. The mandatory IFRS S1/S2 core requirements for the same section
3. The Commercial Banks industry-based guidance where relevant
4. Instructions to satisfy each core requirement and use the industry guidance without treating it as a separate IFRS requirement

### Required Azure `.env` format

This notebook expects the same style of full URL used in your working governance notebook:

```env
AZURE_OPENAI_API_KEY=your_shared_key
AZURE_OPENAI_EXTRACTOR_URL=https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=<api-version>
```

Or reuse your existing judge deployment:

```env
AZURE_OPENAI_API_KEY=your_shared_key
AZURE_OPENAI_JUDGE_URL=https://<resource>.openai.azure.com/openai/deployments/<gpt-5-2-deployment>/chat/completions?api-version=<api-version>
```

### Required source PDFs

Put exactly these three PDFs in `data/standards/`:

```text
ifrs_s1.pdf
ifrs_s2.pdf
ifrs_s2_ibg_volume_16_commercial_banks.pdf
```

The notebook also accepts these alternative file names for convenience:

```text
ifrs-s1-general-requirements.pdf
ifrs-s2-climate-related-disclosures.pdf
ifrs-s2-ibg-volume-16-commercial-banks-part-b.pdf
ifrs-s2-ibg-volume-16-commercial-banks-part-b (1).pdf
```

This notebook now excludes the old generic accompanying guidance and illustrative examples because your selected scope is only:
- IFRS S1
- IFRS S2
- IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks
